# 16 — V5 Fresh DEV + Sealed CONFIRM Roster Freeze

This notebook starts the **data-redesign phase**.

No event outcomes, chronology, astrology features, Control scores, or sealed holdouts are loaded to choose membership.

Targets:

| Split | Competitive | Project | Status | Total |
|---|---:|---:|---:|---:|
| DEV | 40 | 50 | 70 | 160 |
| CONFIRM | 20 | 25 | 35 | 80 |

Only the **DEV** roster may proceed to event collection.  
CONFIRM is frozen now but event collection must remain untouched until a future candidate is frozen.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, unicodedata
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V5_ROSTER_FREEZE_20260817"
SELECTION_SEED = 2026081701

DEV_TARGET = {"COMPETITIVE":40, "PROJECT":50, "STATUS":70}
CONFIRM_TARGET = {"COMPETITIVE":20, "PROJECT":25, "STATUS":35}
FEMALE_MIN_SHARE = 0.20
BIRTH_YEAR_MIN = 1900
BIRTH_YEAR_MAX = 1995

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside the Chartpalja saju repository.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s=unicodedata.normalize("NFKD",str(x))
    s="".join(c for c in s if not unicodedata.combining(c))
    s=s.casefold()
    return re.sub(r"[^a-z0-9]+","",s)

def det_key(name, axis, split):
    raw = "%s|%s|%s|%s" % (norm_name(name), axis, split, SELECTION_SEED)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

ROOT=find_repo_root()

V4_W1_DIR=ROOT/"research/ml/artifacts/v4_unified_dev_roster"
V4_W2_DIR=ROOT/"research/ml/artifacts/v4_unified_dev_wave2_roster"
V4_CORPUS=ROOT/"research/ml_corpus/v4_unified_dev_wave2"

SEED_PATH=V4_CORPUS/"V4_UNIFIED_DEV_WAVE2_PREDECLARED_SEED_UNIVERSE_R3.csv"
BIRTH_PATH=V4_W1_DIR/"PersonList-15k.csv"
W1_PATH=V4_W1_DIR/"V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv"
W2_PATH=V4_W2_DIR/"V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv"
PROTOCOL_PATH=ROOT/"research/ml_corpus/v5_ground_truth/V5_LOCAL_MATCHED_EVENT_PROTOCOL.json"

OUT=ROOT/"research/ml/artifacts/v5_roster_freeze"
OUT.mkdir(parents=True,exist_ok=True)

for p in [SEED_PATH,BIRTH_PATH,W1_PATH,W2_PATH,PROTOCOL_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

seed=pd.read_csv(SEED_PATH)
birth=pd.read_csv(BIRTH_PATH)
w1=pd.read_csv(W1_PATH)
w2=pd.read_csv(W2_PATH)
protocol=json.load(open(PROTOCOL_PATH,encoding="utf-8"))

assert protocol["status"]=="PREDECLARED_BEFORE_V5_EVENT_COLLECTION"
assert protocol["rosters"]["DEV"]["TOTAL"]==160
assert protocol["rosters"]["CONFIRM"]["TOTAL"]==80
assert len(w1)==100 and len(w2)==44

print("Preflight PASS")
print("seed rows:",len(seed))
print("birth rows:",len(birth))


Preflight PASS
seed rows: 915
birth rows: 15807


## 1. Build exclusion universe from consumed development only

In [2]:

def names_from_json(path):
    if not path.exists():
        return []
    try:
        obj=json.load(open(path,encoding="utf-8"))
    except Exception:
        return []
    return [s.get("name") for s in obj.get("subjects",[]) if s.get("name")]

prior_paths={
    "V1":ROOT/"research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json",
    "V2_NEW_DEV":ROOT/"research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json",
    "NEW_DEV_2":ROOT/"research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json",
    "V4_TARGET_EXPANSION":ROOT/"research/ml_corpus/v4_target_expansion/V4_TARGET_EXPANSION_CORPUS.json",
}
prior=[]
for label,p in prior_paths.items():
    ns=names_from_json(p)
    prior.extend(ns)
    print(label,len(ns))

excluded_norm={norm_name(x) for x in prior}
excluded_norm |= set(w1["name"].map(norm_name))
excluded_norm |= set(w2["name"].map(norm_name))

# Explicitly prove sealed datasets are not referenced.
for p in prior_paths.values():
    u=str(p).upper()
    assert "NEW_CONFIRM" not in u
    assert "VALIDATION_B" not in u
    assert "PUBLIC_CHECK" not in u
    assert "PUBLIC_FINAL" not in u

print("excluded unique names:",len(excluded_norm))


V1 94
V2_NEW_DEV 70
NEW_DEV_2 100
V4_TARGET_EXPANSION 169
excluded unique names: 376


## 2. Resolve Rodden-AA births from the frozen snapshot

In [4]:
# 2. Resolve Rodden-AA births from the frozen snapshot
# IMPORTANT:
# This cell does NOT force the original DEV+CONFIRM targets when the current
# role-only seed universe has insufficient fresh AA subjects.
# It freezes the currently available eligible pool and computes deterministic
# expansion deficits. No event / chronology / astrology information is used.

birth = birth.copy()
birth["norm_name"] = birth["Name"].map(norm_name)


def parse_birth_blob(blob):
    obj = json.loads(str(blob))
    std = obj["StdTime"]
    loc = obj["Location"]

    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
        std.strip()
    )

    if not m:
        raise ValueError(std)

    hh, mm, dd, mo, yyyy, offset = m.groups()

    return {
        "birth_date": "%04d-%02d-%02d" % (
            int(yyyy), int(mo), int(dd)
        ),
        "birth_time": "%02d:%02d" % (
            int(hh), int(mm)
        ),
        "utc_offset": offset,
        "birth_year": int(yyyy),
        "birth_place": loc["Name"],
        "longitude": float(loc["Longitude"]),
        "latitude": float(loc["Latitude"]),
    }


rows = []

for _, r in birth.iterrows():

    # Keep Rodden AA only where an explicit rating field exists.
    grade = str(
        r.get(
            "RoddenRating",
            r.get(
                "Rodden",
                r.get("Rating", "")
            )
        )
    ).strip().upper()

    if grade and grade != "AA":
        continue

    try:
        p = parse_birth_blob(r["BirthTime"])
    except Exception:
        continue

    rows.append({
        "source_row_key": r.get("RowKey"),
        "source_name": r.get("Name"),
        "gender": str(r.get("Gender", "")).lower(),
        "norm_name": r["norm_name"],
        **p,
    })


bp = pd.DataFrame(rows)

# Never silently resolve duplicate names.
counts = bp.groupby("norm_name").size()
ambiguous_names = set(
    counts[counts > 1].index
)

bp = bp[
    ~bp.norm_name.isin(ambiguous_names)
].copy()


# ---------------------------------------------------------
# Resolve the PREDECLARED role-only seed universe.
# ---------------------------------------------------------

seed = seed.copy()
seed["norm_name"] = seed["name"].map(norm_name)

assert (
    seed.groupby("norm_name")["axis"]
    .nunique()
    .max()
    == 1
)


resolved = seed.merge(
    bp,
    on="norm_name",
    how="left",
    suffixes=("", "_birth")
)


resolved["prior_used"] = (
    resolved["norm_name"]
    .isin(excluded_norm)
)

resolved["birth_ok"] = (
    resolved["birth_year"]
    .between(
        BIRTH_YEAR_MIN,
        BIRTH_YEAR_MAX,
        inclusive="both"
    )
)

resolved["eligible"] = (
    resolved["source_row_key"].notna()
    & ~resolved["prior_used"]
    & resolved["birth_ok"]
)


summary = (
    resolved
    .groupby("axis")
    .agg(
        seeds=("name", "size"),
        birth_resolved=(
            "source_row_key",
            lambda x: x.notna().sum()
        ),
        prior_used=("prior_used", "sum"),
        eligible=("eligible", "sum"),
    )
    .reset_index()
)

display(summary)


# ---------------------------------------------------------
# IMPORTANT CHANGE:
# Do NOT fail merely because the current seed universe
# cannot satisfy the final V5 targets.
#
# Instead calculate what a NEW ROLE-ONLY seed expansion
# must provide.
# ---------------------------------------------------------

FINAL_DEV_TARGET = {
    "COMPETITIVE": 40,
    "PROJECT": 50,
    "STATUS": 70,
}

FINAL_CONFIRM_TARGET = {
    "COMPETITIVE": 20,
    "PROJECT": 25,
    "STATUS": 35,
}


available = {
    axis: int(
        summary.loc[
            summary.axis == axis,
            "eligible"
        ].iloc[0]
    )
    for axis in FINAL_DEV_TARGET
}


required_total = {
    axis: (
        FINAL_DEV_TARGET[axis]
        + FINAL_CONFIRM_TARGET[axis]
    )
    for axis in FINAL_DEV_TARGET
}


expansion_deficit = {
    axis: max(
        0,
        required_total[axis]
        - available[axis]
    )
    for axis in required_total
}


# Also report DEV-only deficit.
# This tells us what must exist before DEV event collection
# is allowed to start.
dev_deficit = {
    axis: max(
        0,
        FINAL_DEV_TARGET[axis]
        - available[axis]
    )
    for axis in FINAL_DEV_TARGET
}


availability_table = pd.DataFrame([
    {
        "axis": axis,
        "fresh_AA_available_now": available[axis],
        "DEV_target": FINAL_DEV_TARGET[axis],
        "CONFIRM_target": FINAL_CONFIRM_TARGET[axis],
        "DEV_plus_CONFIRM_target": required_total[axis],
        "DEV_deficit": dev_deficit[axis],
        "total_expansion_deficit": expansion_deficit[axis],
    }
    for axis in FINAL_DEV_TARGET
])

display(availability_table)


# Freeze the currently available pool BEFORE any event work.
wave_a_pool = (
    resolved[
        resolved["eligible"]
    ]
    .copy()
)

wave_a_pool["wave"] = "V5_ROSTER_WAVE_A_AVAILABLE_POOL"
wave_a_pool["event_collection_started"] = False
wave_a_pool["astrology_scored"] = False

wave_a_path = (
    OUT
    / "V5_ROSTER_WAVE_A_AVAILABLE_POOL.csv"
)

wave_a_pool.to_csv(
    wave_a_path,
    index=False
)


expansion_request = {
    "version": "V5_ROSTER_EXPANSION_REQUEST_V1",
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "status": (
        "V5_CURRENT_SEED_UNIVERSE_INSUFFICIENT_"
        "EXPANSION_REQUIRED_BEFORE_FINAL_ROSTER_FREEZE"
    ),

    "available_fresh_AA": available,

    "final_DEV_target": FINAL_DEV_TARGET,
    "final_CONFIRM_target": FINAL_CONFIRM_TARGET,

    "DEV_deficit": dev_deficit,
    "DEV_plus_CONFIRM_expansion_deficit": (
        expansion_deficit
    ),

    "wave_a_pool_sha256": (
        sha256_file(wave_a_path)
    ),

    "rules": {
        "event_collection_allowed": False,
        "astrology_generation_allowed": False,
        "prior_subject_reuse_allowed": False,
        "target_reduction_allowed": False,

        "next_seed_expansion_may_use": [
            "role/category",
            "identity",
            "birth availability",
            "Rodden AA",
            "birth year",
            "gender guardrail",
        ],

        "next_seed_expansion_may_not_use": [
            "event outcome",
            "event chronology",
            "pairability",
            "astrology",
            "Control score",
        ],
    },
}


with open(
    OUT / "V5_ROSTER_EXPANSION_REQUEST.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        expansion_request,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        expansion_request,
        ensure_ascii=False,
        indent=2
    )
)

print()
print(
    "STOP HERE: do not run the roster-selection "
    "cells below yet."
)

,axis,seeds,birth_resolved,prior_used,eligible
0,COMPETITIVE,304,59,14,45
1,PROJECT,193,76,24,50
2,STATUS,418,69,59,14


,axis,fresh_AA_available_now,DEV_target,CONFIRM_target,DEV_plus_CONFIRM_target,DEV_deficit,total_expansion_deficit
0,COMPETITIVE,45,40,20,60,0,15
1,PROJECT,50,50,25,75,0,25
2,STATUS,14,70,35,105,56,91


{
  "version": "V5_ROSTER_EXPANSION_REQUEST_V1",
  "created_at": "2026-08-17T02:06:54",
  "status": "V5_CURRENT_SEED_UNIVERSE_INSUFFICIENT_EXPANSION_REQUIRED_BEFORE_FINAL_ROSTER_FREEZE",
  "available_fresh_AA": {
    "COMPETITIVE": 45,
    "PROJECT": 50,
    "STATUS": 14
  },
  "final_DEV_target": {
    "COMPETITIVE": 40,
    "PROJECT": 50,
    "STATUS": 70
  },
  "final_CONFIRM_target": {
    "COMPETITIVE": 20,
    "PROJECT": 25,
    "STATUS": 35
  },
  "DEV_deficit": {
    "COMPETITIVE": 0,
    "PROJECT": 0,
    "STATUS": 56
  },
  "DEV_plus_CONFIRM_expansion_deficit": {
    "COMPETITIVE": 15,
    "PROJECT": 25,
    "STATUS": 91
  },
  "wave_a_pool_sha256": "35ac1a96c66b89299ac5140f45ec11c0ca2cb8088ee5efcca3fd0c0264a08b69",
  "rules": {
    "event_collection_allowed": false,
    "astrology_generation_allowed": false,
    "prior_subject_reuse_allowed": false,
    "target_reduction_allowed": false,
    "next_seed_expansion_may_use": [
      "role/category",
      "identity",
      "bir

## 3. Deterministically freeze DEV first, then CONFIRM from the remainder

In [6]:
# 3. Final roster selection gate
#
# The current role-only seed universe is insufficient,
# especially for STATUS.
#
# Therefore DEV / CONFIRM membership MUST NOT be frozen yet.
# Wait for a new role-only seed expansion wave.

EXPANSION_REQUEST_PATH = (
    OUT / "V5_ROSTER_EXPANSION_REQUEST.json"
)

WAVE_A_PATH = (
    OUT / "V5_ROSTER_WAVE_A_AVAILABLE_POOL.csv"
)

if not EXPANSION_REQUEST_PATH.exists():
    raise FileNotFoundError(
        "Run Cell 2 first. "
        "V5_ROSTER_EXPANSION_REQUEST.json is missing."
    )

if not WAVE_A_PATH.exists():
    raise FileNotFoundError(
        "Run Cell 2 first. "
        "V5_ROSTER_WAVE_A_AVAILABLE_POOL.csv is missing."
    )


with open(
    EXPANSION_REQUEST_PATH,
    encoding="utf-8"
) as f:
    expansion_request = json.load(f)


expected_status = (
    "V5_CURRENT_SEED_UNIVERSE_INSUFFICIENT_"
    "EXPANSION_REQUIRED_BEFORE_FINAL_ROSTER_FREEZE"
)

assert (
    expansion_request["status"]
    == expected_status
), expansion_request["status"]


print("=" * 80)
print("V5 FINAL ROSTER FREEZE: PAUSED")
print("=" * 80)

print()
print("Current fresh AA availability:")
print(
    json.dumps(
        expansion_request["available_fresh_AA"],
        ensure_ascii=False,
        indent=2
    )
)

print()
print("Additional fresh subjects required for DEV + CONFIRM:")
print(
    json.dumps(
        expansion_request[
            "DEV_plus_CONFIRM_expansion_deficit"
        ],
        ensure_ascii=False,
        indent=2
    )
)

print()
print(
    "STATUS: "
    "V5_WAITING_FOR_ROLE_ONLY_SEED_EXPANSION"
)

print()
print(
    "DO NOT freeze DEV / CONFIRM yet."
)
print(
    "DO NOT collect events yet."
)
print(
    "DO NOT generate astrology yet."
)

print()
print(
    "Next step: create a completely new role-only "
    "candidate expansion and resolve fresh Rodden-AA births."
)


V5_ROSTER_STAGE = (
    "WAITING_FOR_ROLE_ONLY_SEED_EXPANSION"
)

V5 FINAL ROSTER FREEZE: PAUSED

Current fresh AA availability:
{
  "COMPETITIVE": 45,
  "PROJECT": 50,
  "STATUS": 14
}

Additional fresh subjects required for DEV + CONFIRM:
{
  "COMPETITIVE": 15,
  "PROJECT": 25,
  "STATUS": 91
}

STATUS: V5_WAITING_FOR_ROLE_ONLY_SEED_EXPANSION

DO NOT freeze DEV / CONFIRM yet.
DO NOT collect events yet.
DO NOT generate astrology yet.

Next step: create a completely new role-only candidate expansion and resolve fresh Rodden-AA births.


## 4. Freeze files + DEV-only event intake template

In [ ]:

dev_path=OUT/"V5_DEV_SUBJECT_ROSTER_160.csv"
confirm_path=OUT/"V5_CONFIRM_SUBJECT_ROSTER_80_SEALED.csv"
dev.to_csv(dev_path,index=False)
confirm.to_csv(confirm_path,index=False)

event_cols=[
    "subject_id","name","preassigned_axis",
    "event_year","polarity","event_type","event_description",
    "source_url","source_title","source_publisher","source_date",
    "source_quality","eligibility_note","exclude","exclude_reason"
]
template=dev[["subject_id","name","preassigned_axis"]].copy()
for c in event_cols[3:]:
    template[c]=""
template.to_csv(OUT/"V5_DEV_EVENT_INTAKE_TEMPLATE.csv",index=False)

freeze={
    "version":"V5_ROSTER_FREEZE_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_DEV_AND_CONFIRM_ROSTERS_FROZEN_READY_FOR_DEV_EVENT_COLLECTION",
    "dev_n":len(dev),
    "confirm_n":len(confirm),
    "dev_axis_counts":dev.preassigned_axis.value_counts().to_dict(),
    "confirm_axis_counts":confirm.preassigned_axis.value_counts().to_dict(),
    "dev_roster_sha256":sha256_file(dev_path),
    "confirm_roster_sha256":sha256_file(confirm_path),
    "protocol_sha256":sha256_file(PROTOCOL_PATH),
    "seed_sha256":sha256_file(SEED_PATH),
    "birth_snapshot_sha256":sha256_file(BIRTH_PATH),
    "selection_seed":SELECTION_SEED,
    "event_collection_started":False,
    "astrology_scored":False,
    "confirm_event_collection_allowed":False,
    "membership_used_event_outcomes":False,
    "membership_used_chronology":False,
    "membership_used_astrology":False,
    "sealed_holdouts_loaded":False,
    "next_rule":"Collect events for V5 DEV 160 only under V5_LOCAL_MATCHED_EVENT_PROTOCOL. Do not research CONFIRM events."
}
json.dump(freeze,open(OUT/"V5_ROSTER_FREEZE_DECISION.json","w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(freeze,ensure_ascii=False,indent=2))


## Send back

After `Kernel Restart -> Run All`, send exactly:

```text
V5_ROSTER_FREEZE_DECISION.json
V5_DEV_SUBJECT_ROSTER_160.csv
V5_DEV_EVENT_INTAKE_TEMPLATE.csv
```

Do **not** send or inspect the CONFIRM event layer because it must not exist yet.  
The confirmation roster file may remain sealed in the repo.
